# WWTD-2025 (What Would Trump Do?)

Generate a forecasting dataset about Trump's actions, decisions, and statements using the LightningRod SDK. This example showcases dataset generation, preparation with SDK utils, and training results from our experiments—including evaluation with and without context.

In [1]:
%pip install lightningrod-ai python-dotenv pandas openai

from IPython.display import clear_output
clear_output()

from datetime import datetime

import pandas as pd
from dotenv import load_dotenv

load_dotenv()

True

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/?redirect=/api) to get your API key and **$50 of free credits**.

In [2]:
from lightningrod import LightningRod
from lightningrod.utils import config

api_key = config.get_config_value("LIGHTNINGROD_API_KEY")
lr = LightningRod(api_key=api_key)

## Build the pipeline

Configure the pipeline with domain-specific instructions and examples for Trump-related forecasting.

In [3]:
instructions = """
Generate binary forecasting questions about Trump's actions, decisions, positions, and statements.
Questions should be diverse, related to the content, and should evenly cover the full range from very likely to very unlikely.
Horizon: outcomes should be known within 2 months of the question date, and may be known much sooner.
Criteria: binary outcome, exact dates, self-contained, verifiable via web search, newsworthy.
"""

good_examples = [
    "Will Trump impose 25% tariffs on all goods from Canada by February 1, 2025?",
    "Will Trump issue pardons to January 6 defendants within his first week in office?",
    "Will Pete Hegseth be confirmed as Secretary of Defense by February 15, 2025?",
    "Will Trump sign an executive order to keep TikTok operational in the US by January 31, 2025?",
    "Will Kash Patel be confirmed as FBI Director by March 1, 2025?",
]

bad_examples = [
    "Will Trump do something controversial? (too vague)",
    "Will Trump be in the news? (obvious)",
    "Will tariffs be imposed? (needs specifics)",
]

In [4]:
from lightningrod import (
    BinaryAnswerType,
    NewsSeedGenerator,
    ForwardLookingQuestionGenerator,
    NewsContextGenerator,
    WebSearchLabeler,
    QuestionPipeline,
)

answer_type = BinaryAnswerType()

pipeline = QuestionPipeline(
    seed_generator=NewsSeedGenerator(
        start_date=datetime(2025, 1, 1),
        end_date=datetime(2026, 1, 1),
        interval_duration_days=7,
        search_query=[
            "Donald Trump domestic policy agenda",
            "Donald Trump trade and tariff actions",
            "Donald Trump foreign policy decisions",
            "Donald Trump interviews and press appearances",
            "Donald Trump lawsuits and court rulings",
        ],
        articles_per_search=10,
    ),
    question_generator=ForwardLookingQuestionGenerator(
        instructions=instructions,
        examples=good_examples,
        bad_examples=bad_examples,
        answer_type=answer_type,
        questions_per_seed=5,
    ),
    context_generators=[
        NewsContextGenerator(
            articles_per_query=3,
            num_search_queries=1,
            num_articles=5,
        )
    ],
    labeler=WebSearchLabeler(answer_type=answer_type),
)

## Run the pipeline

This will collect news articles, generate questions, and find answers. Use `max_seeds` to limit the run for testing.

In [5]:
dataset = lr.transforms.run(pipeline, max_seeds=100, name="WWTD-2025")  # Increase to ~2000 for a real run

samples = dataset.download()
pct = (sum(1 for s in samples if s.is_valid is True) / len(samples) * 100) if samples else 0
print(f"{len(samples)} samples ({pct:.1f}% valid)")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Pipeline Completed                                                                                          │
│                                                                                                                 │
│    Total cost: $1.90                                                                                            │
│                                                                                                                 │
│  ┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━┳━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓  │
│  ┃ Step               ┃ Progress             ┃  In ┃ Out ┃ Rejected ┃ Errors ┃ Rejection Reasons  ┃ Duration ┃  │
│  ┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━╇━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩  │
│  │ NewsSeedGenerator… │ Complete             │  10 │  95 │        0 │      0 │ -                  │       1s │  │
│  │ ForwardLookingQue… │ Complete             │  95 │ 442 │       31 │      0 │ date_close not     │       1s │  │
│  │                    │                      │     │     │          │        │ after event_date   │          │  │
│  │                    │                      │     │     │          │        │ (31)               │          │  │
│  │ WebSearchLabelerT… │ Complete             │ 442 │ 380 │       62 │      0 │ Resolution date is │       4s │  │
│  │                    │                      │     │     │          │        │ before seed        │          │  │
│  │                    │                      │     │     │          │        │ creation date      │          │  │
│  │                    │                      │     │     │          │        │ (36), Undetermined │          │  │
│  │                    │                      │     │     │          │        │ label (25), Low    │          │  │
│  │                    │                      │     │     │          │        │ confidence: 0.80 < │          │  │
│  │                    │                      │     │     │          │        │ 0.9 (1)            │          │  │
│  │ NewsContextGenera… │ Complete             │ 380 │ 380 │        0 │      0 │ -                  │      57s │  │
│  └────────────────────┴──────────────────────┴─────┴─────┴──────────┴────────┴────────────────────┴──────────┘  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

473 samples (80.3% valid)


## Prepare the dataset

Use SDK utils to filter valid samples, deduplicate, and split into train/test sets. We filter by `date_close <= today` to only include questions that have already resolved.

In [6]:
from lightningrod import prepare_for_training, FilterParams, SplitParams

train_dataset, test_dataset = prepare_for_training(
    dataset,
    filter=FilterParams(days_to_resolution_range=(1, 60)),
    split=SplitParams(test_size=0.2),
)

for name, ds in [("Train", train_dataset), ("Test", test_dataset)]:
    data = ds.flattened()
    print(len(data))
    yes_count = sum(1 for s in data if s.get("label") in (1, "1", 1.0))
    print(f"{name}: {len(data)} rows, {yes_count/len(data)*100:.1f}% yes")
    display(pd.DataFrame(data).head())

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> prepare_for_training                                                                                        │
│                                                                                                                 │
│    Starting with 473 samples                                                                                    │
│                                                                                                                 │
│    Filter:  Dropped 93 invalid, 103 horizon → 277 remain                                                        │
│    Dedup:   277 remain (0 duplicates)                                                                           │
│    Split:   Splits: 167 train | 56 test (0 dropped, no prediction_date)                                         │
│             54 train samples removed for leakage                                                                │
│                                                                                                                 │
│  ⚠ Unhealthy dataset                                                                                            │
│                                                                                                                 │
│  Only 167 train samples remain after preparation. This is below the recommended minimum of 200 for effective    │
│  training.                                                                                                      │
│                                                                                                                 │
│    Tips:                                                                                                        │
│      • Increase max_questions in lr.transforms.run() to generate more samples.                                  │
│      • Increase questions_per_seed in your question generator (ForwardLookingQuestionGenerator or               │
│  QuestionGenerator) to produce more questions from each seed article.Add more search queries to your seed       │
│  generator to diversify seed sources.                                                                           │
│      • Widen the seed generator date range (start_date to end_date) to capture more events.                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

167
Train: 167 rows, 21.0% yes


,sample_id,is_valid,question_text,date_close,event_date,resolution_criteria,prediction_date,label,answer_type,label_confidence,...,reasoning,answer_sources,seed_text,seed_url,seed_creation_date,seed_search_query,context,meta_sample_id,meta_parent_sample_id,meta_processing_time_ms
0,07511299-78d6-4020-8efe-d7b5b865f826,True,Will the 11th Circuit Court of Appeals issue a...,2025-02-15T00:00:00,2025-01-08T00:00:00,This question resolves to 'Yes' if the U.S. Co...,2025-01-08T00:00:00,1,binary,1.00,...,"On January 9, 2025, the U.S. Court of Appeals ...",https://vertexaisearch.cloud.google.com/ground...,Title: The Situation: Ending the Trump Cases t...,https://www.lawfaremedia.org/article/the-situa...,2025-01-08T00:00:00,Donald Trump lawsuits and court rulings,[{'rendered_context': '--- ARTICLES [1] Judge ...,59824a03-d075-4de4-bd89-17ec5f0651f1,da9d2197-889d-4fe4-ae4e-960ab3f9726f,16820.019
1,3618250c-9aa7-4e3e-b1f4-d59351e25415,True,Will the criminal charges against Carlos De Ol...,2025-03-05T00:00:00,2025-01-08T00:00:00,This question resolves to 'Yes' if a federal c...,2025-01-08T00:00:00,1,binary,1.00,...,The criminal charges against Carlos De Oliveir...,https://vertexaisearch.cloud.google.com/ground...,Title: The Situation: Ending the Trump Cases t...,https://www.lawfaremedia.org/article/the-situa...,2025-01-08T00:00:00,Donald Trump lawsuits and court rulings,[{'rendered_context': '--- ARTICLES [1] Trump ...,89a53581-5832-4797-9a76-4a85aefb8993,da9d2197-889d-4fe4-ae4e-960ab3f9726f,170592.880
2,6f5808f8-5fbe-40e3-a902-2959e0159960,True,Will Justice Juan Merchan sentence Donald Trum...,2025-03-01T00:00:00,2025-01-08T00:00:00,This question resolves to 'Yes' if Justice Jua...,2025-01-08T00:00:00,0,binary,1.00,...,The close date for this question is 2025-03-01...,https://vertexaisearch.cloud.google.com/ground...,Title: The Situation: Ending the Trump Cases t...,https://www.lawfaremedia.org/article/the-situa...,2025-01-08T00:00:00,Donald Trump lawsuits and court rulings,[{'rendered_context': '--- ARTICLES [1] Judge ...,da9d2197-889d-4fe4-ae4e-960ab3f9726f,b988692d-28ac-4e59-932f-089b30c1fdff,19099.822
3,811942a3-0ce0-4a23-8ccf-cb9b6b038a78,True,"Will the full, unredacted Special Counsel repo...",2025-03-01T00:00:00,2025-01-08T00:00:00,This question resolves to 'Yes' if the Departm...,2025-01-08T00:00:00,0,binary,1.00,...,Special Counsel Jack Smith submitted a two-vol...,https://vertexaisearch.cloud.google.com/ground...,Title: The Situation: Ending the Trump Cases t...,https://www.lawfaremedia.org/article/the-situa...,2025-01-08T00:00:00,Donald Trump lawsuits and court rulings,[{'rendered_context': '--- ARTICLES [1] Trump ...,d53c18aa-e58f-47cc-ad8d-07284f3837b5,da9d2197-889d-4fe4-ae4e-960ab3f9726f,210318.406
4,abbac7a6-09fd-4d34-b588-0a3df04f2f37,True,Will Donald Trump grant a formal presidential ...,2025-02-28T00:00:00,2025-01-08T00:00:00,This question resolves to 'Yes' if the White H...,2025-01-08T00:00:00,0,binary,0.95,...,The close date for this question is 2025-02-28...,https://vertexaisearch.cloud.google.com/ground...,Title: The Situation: Ending the Trump Cases t...,https://www.lawfaremedia.org/article/the-situa...,2025-01-08T00:00:00,Donald Trump lawsuits and court rulings,"[{'rendered_context': '', 'search_query': 'Tru...",c1f9b0fa-f364-4cda-a9ca-a41f1dcdcf3c,da9d2197-889d-4fe4-ae4e-960ab3f9726f,16623.478


56
Test: 56 rows, 37.5% yes


,sample_id,is_valid,question_text,date_close,event_date,resolution_criteria,prediction_date,label,answer_type,label_confidence,...,reasoning,answer_sources,seed_text,seed_url,seed_creation_date,seed_search_query,context,meta_sample_id,meta_parent_sample_id,meta_processing_time_ms
0,7960bd90-44c8-4cd0-bac0-3a17265101e5,True,Will Donald Trump announce a complete exemptio...,2025-12-01T00:00:00,2025-10-08T00:00:00,The question is answered 'Yes' if the US Presi...,2025-10-08T00:00:00,0,binary,1.00,...,The close date for this question is 2025-12-01...,https://vertexaisearch.cloud.google.com/ground...,"Title: 'We will get an even better deal,' Carn...",https://www.cbc.ca/news/politics/carney-even-b...,2025-10-08T00:00:00,Donald Trump trade and tariff actions,[{'rendered_context': '--- ARTICLES [1] U.S.-C...,ecd757d7-fa3c-47ec-bdc6-81cdf88c1b65,a3db9248-0a4f-4d51-81d1-7c21057856cf,12056.629
1,b562085f-b58d-4284-ae0b-fcfdeb90f90c,True,Will the United States and Canada sign a forma...,2025-12-01T00:00:00,2025-10-08T00:00:00,A formal bilateral agreement or signed Memoran...,2025-10-08T00:00:00,0,binary,0.95,...,Based on the provided reports regarding the 20...,https://vertexaisearch.cloud.google.com/ground...,"Title: 'We will get an even better deal,' Carn...",https://www.cbc.ca/news/politics/carney-even-b...,2025-10-08T00:00:00,Donald Trump trade and tariff actions,[{'rendered_context': '--- ARTICLES [1] Carney...,a3db9248-0a4f-4d51-81d1-7c21057856cf,993816e2-887d-4db0-99c7-f45e93776a8a,255122.468
2,63dc110c-6073-4f98-abcb-57b8933b8fbf,True,Will the Trump administration hold a new offsh...,2026-04-01T00:00:00,2025-11-26T00:00:00,The question resolves to 'Yes' if the Departme...,2025-11-26T00:00:00,1,binary,1.00,...,The Trump administration held the first new of...,https://vertexaisearch.cloud.google.com/ground...,Title: nytimes.com\n\nURL Source: https://www....,https://www.nytimes.com/2025/11/26/climate/tru...,2025-11-26T00:00:00,Donald Trump domestic policy agenda,[{'rendered_context': '--- ARTICLES [1] Interi...,87e57caa-9696-4eb8-a7ed-cd4060b0649e,b4bfbb84-63db-48d6-b63d-c809b20ed5f0,6996.226
3,2581c035-5519-434f-aed1-c51326b11e73,True,Will Donald Trump and Javier Milei hold a join...,2026-01-01T00:00:00,2025-11-27T00:00:00,The question resolves to 'Yes' if Donald Trump...,2025-11-27T00:00:00,0,binary,0.95,...,The close date for this question is 2026-01-01...,https://vertexaisearch.cloud.google.com/ground...,Title: The Paradox of Europe's Trumpian Right:...,https://www.foreignaffairs.com/europe/paradox-...,2025-11-27T00:00:00,Donald Trump domestic policy agenda,[{'rendered_context': '--- ARTICLES [1] Milei ...,3a88f43d-6a2c-422f-b7dd-5b6039a29375,7977998c-8ccb-444f-a382-1fcd8e322c23,11967.560
4,3bd13f7e-25a8-40c0-8b2a-d9f77ef9f827,True,Will the United States official executive bran...,2026-01-20T00:00:00,2025-11-27T00:00:00,The question resolves to 'Yes' if the U.S. gov...,2025-11-27T00:00:00,1,binary,0.95,...,Between the question date (2025-11-27) and the...,https://vertexaisearch.cloud.google.com/ground...,Title: The Paradox of Europe's Trumpian Right:...,https://www.foreignaffairs.com/europe/paradox-...,2025-11-27T00:00:00,Donald Trump domestic policy agenda,[{'rendered_context': '--- ARTICLES [1] Presid...,d15a72f1-d8ac-4fc9-9bde-732514e9008c,7977998c-8ccb-444f-a382-1fcd8e322c23,66899.224


## Model Training

Fine-tune a forecasting model on your dataset. For production training, generate more questions (increase `max_seeds` or run without limit). Our reference experiments used 2,790 questions—see [Trump-Forecaster Model](https://huggingface.co/LightningRodLabs/Trump-Forecaster) and [Trump-Forecaster Dataset](https://huggingface.co/datasets/LightningRodLabs/WWTD-2025) for details.

## Estimate training cost

Before starting a job, use `estimate_cost` to see the expected cost and token usage.

In [7]:
from lightningrod import GRPOTrainingConfig

config = GRPOTrainingConfig(
    base_model_id="openai/gpt-oss-120b",
    training_steps=50,
)
cost_estimate = lr.training.estimate_cost(config, dataset=train_dataset)
print(f"Estimated cost: ${cost_estimate.total_cost_dollars:.2f}")
print(f"Effective steps: {cost_estimate.effective_steps}")
print(f"Train tokens: {cost_estimate.train_tokens:,}")
print(f"Notes: {cost_estimate.notes}")

Estimated cost: $0.18
Effective steps: 6
Train tokens: 617,089
Notes: Estimate uses per-answer-type output token estimates; actual may vary


## Start training

`run` creates a job and polls until completion with a live progress display.

In [8]:
job = lr.training.run(config, dataset=train_dataset, name="WWTD-2025")
print(f"Job {job.id} completed with status: {job.status}")
print(f"Trained model ID: {job.model_id}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Training COMPLETED                                                                                          │
│                                                                                                                 │
│    Job: WWTD-2025                                                                                               │
│                                                                                                                 │
│    Reward: latest -1.3368  avg -0.8626  (6 steps)  (higher is better)                                           │
│                                                                                                                 │
│    Cost:  $0.11                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Job 371e5ff4-ebf5-43af-8809-d26c14edb8e2 completed with status: COMPLETED
Trained model ID: checkpoint:371e5ff4-ebf5-43af-8809-d26c14edb8e2


## Inference with your trained model

Use `lr.predict()` to run inference with your trained model.

In [ ]:
print(lr.predict(job.model_id, "Will Trump impose 25% tariffs on all goods from Canada by February 1, 2027?"))

## Run evals on trained model

Run test evals on your trained model against the test dataset. The eval job runs the model on the dataset and reports metrics.

In [ ]:
from lightningrod import EvalModel, training

eval_job = lr.evals.run_from_training_job(
    config,
    job,
    test_dataset,
    extra_models=[
        EvalModel(model_id="openai/gpt-5.2", label="GPT-5.2"),
    ],
)

training.print_eval(eval_job)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Eval COMPLETED                                                                                              │
│                                                                                                                 │
│    ID: 00602447-5872-4732-93f9-b0d99459da1a                                                                     │
│    Model: checkpoint:13fa02ec-27f4-47a9-84c9-762d91a1904a                                                       │
│    Dataset: 82186c26-a309-43a6-9543-37bdda38d41d                                                                │
│                                                                                                                 │
│  ┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━┓                                                        │
│  ┃ Metric              ┃    base ┃ trained ┃ benchmark ┃                                                        │
│  ┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━┩                                                        │
│  │ brier_score         │  0.2333 │  0.1877 │    0.1555 │                                                        │
│  │ ece                 │  0.1451 │  0.0963 │    0.0590 │                                                        │
│  │ mean_reward         │ -0.7840 │ -0.6048 │   -0.4966 │                                                        │
│  │ mean_valid_reward   │ -0.7840 │ -0.6048 │   -0.4966 │                                                        │
│  │ n_samples           │     113 │     113 │       113 │                                                        │
│  │ n_valid             │     113 │     113 │       113 │                                                        │
│  │ parse_rate          │  1.0000 │  1.0000 │    1.0000 │                                                        │
│  │ total_cost          │  0.0068 │  0.0068 │         — │                                                        │
│  │ total_input_tokens  │   93344 │   93344 │     88060 │                                                        │
│  │ total_output_tokens │    1111 │    1101 │     28947 │                                                        │
│  └─────────────────────┴─────────┴─────────┴───────────┘                                                        │
│                                                                                                                 │
│    Cost:  $0.01                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

> Note: the trained model checkpoint will only be available for 7 days. If you wish to host this model long-term, reach out to us at support@lightningrod.ai.